<a href="https://colab.research.google.com/github/ishwarraja/SOAI/blob/main/ERAv4/S7/S7_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Depthwise separable block
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, padding=1, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size=k, stride=stride,
                            padding=padding, dilation=dilation,
                            groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = self.bn(x)
        return self.act(x)

class ModelC1C2C3C4(nn.Module):
    def __init__(self, base_ch=20, num_classes=10, dropout_p=0.1):
        super().__init__()
        C = base_ch

        # C1: simple conv
        self.c1 = nn.Sequential(
            nn.Conv2d(3, C, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )

        # C2: depthwise separable conv
        self.c2 = DepthwiseSeparableConv(C, C*2, k=3, stride=1, padding=1)
        C = C*2

        # C3: dilated stack to expand RF smoothly (dilations 1->2->4)
        self.c3_1 = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=1, padding=1, dilation=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )
        self.c3_2 = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=1, padding=2, dilation=2, bias=False),  # dilated
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )
        self.c3_3 = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=1, padding=4, dilation=4, bias=False),  # dilated
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )
        self.c3_4 = nn.Sequential(
            nn.Conv2d(40, 40, kernel_size=3, stride=1, padding=8, dilation=8, bias=False),
            nn.BatchNorm2d(40),
            nn.ReLU(inplace=True)
        )

        self.c3_5 = nn.Sequential(
            nn.Conv2d(40, 40, kernel_size=3, stride=1, padding=16, dilation=16, bias=False),
            nn.BatchNorm2d(40),
            nn.ReLU(inplace=True)
        )

        # C4: downsample via conv stride=2 (no MaxPool)
        self.c4 = nn.Sequential(
            nn.Conv2d(C, C*2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(C*2),
            nn.ReLU(inplace=True)
        )
        C = C*2

        # small head and dropout before GAP
        self.head = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(C, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if getattr(m, 'bias', None) is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.c1(x)          # C1
        x = self.c2(x)          # C2 depthwise separable
        x = self.c3_1(x)        # C3 dilation 1
        x = self.c3_2(x)        # C3 dilation 2
        x = self.c3_3(x)        # C3 dilation 4
        x = self.c4(x)          # C4 stride=2
        x = self.head(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [6]:
# Run this cell in Colab (after the model cell)
from torchsummary import summary
import torch, torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
model = ModelC1C2C3C4(base_ch=20).to(device)

# torchsummary sometimes errors on non-CPU devices; safely run on CPU and return model to device
try:
    summary(model.to("cpu"), (3,32,32))
    model.to(device)
except Exception as e:
    print("torchsummary fallback:", e)
    # fallback: print param count and conv list
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total params (fallback):", total_params)
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            print(name, module)

# exact parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable params: {total_params:,}")

# RF calculator
def calc_rf(m):
    rf = 1
    jump = 1
    for layer in m.modules():
        if isinstance(layer, nn.Conv2d):
            k = layer.kernel_size[0]
            s = layer.stride[0]
            d = layer.dilation[0]
            rf = rf + (k-1)*d*jump
            jump *= s
    return rf

print("Estimated RF:", calc_rf(model))

# Quick checks
assert total_params < 200_000, "Params exceed 200k"
assert calc_rf(model) > 44, "RF not > 44"


torchsummary fallback: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same
Total params (fallback): 161570
c1.0 Conv2d(3, 20, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
c2.dw Conv2d(20, 20, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=20, bias=False)
c2.pw Conv2d(20, 40, kernel_size=(1, 1), stride=(1, 1), bias=False)
c3_1.0 Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
c3_2.0 Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2), bias=False)
c3_3.0 Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(4, 4), dilation=(4, 4), bias=False)
c3_4.0 Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(8, 8), dilation=(8, 8), bias=False)
c3_5.0 Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(16, 16), dilation=(16, 16), bias=False)
c4.0 Conv2d(40, 80, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
head.0 Conv2d(80, 80, kernel_

In [7]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    # ShiftScaleRotate - keeps the functionality required
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.6),
    # CoarseDropout per assignment (use fill_value = CIFAR mean in 0..255)
    A.CoarseDropout(
        max_holes=1,
        max_height=16, max_width=16,
        min_holes=1, min_height=16, min_width=16,
        fill_value=tuple(int(255.0*m) for m in mean),
        mask_fill_value=None,
        p=0.5
    ),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-2407694874.py:13: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value, mask_fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


In [8]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CIFAR10
import numpy as np
from tqdm import tqdm

# Dataset wrapper using albumentations
class CIFARDataset(Dataset):
    def __init__(self, train=True, transform=None):
        self.ds = CIFAR10(root='./data', train=train, download=True)
        self.transform = transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img = self.ds.data[idx]  # HWC uint8
        label = int(self.ds.targets[idx])
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, pred, target):
        num_classes = pred.size(1)
        logprobs = F.log_softmax(pred, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(logprobs)
            true_dist.fill_(self.smoothing / (num_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * logprobs, dim=1))

# Instantiate data loaders
train_ds = CIFARDataset(train=True, transform=train_transforms)
val_ds   = CIFARDataset(train=False, transform=valid_transforms)
batch = 256  # reduce if GPU OOM
train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=batch, shuffle=False, num_workers=4, pin_memory=True)

# Model & device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModelC1C2C3C4(base_ch=20).to(device)

# Opt & scheduler
epochs = 300
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4, nesterov=True)

# OneCycleLR: set steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.2,
                                                steps_per_epoch=len(train_loader), epochs=epochs)

criterion = LabelSmoothingLoss(smoothing=0.1)

best = 0.0
for ep in range(1, epochs+1):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for xb, yb in tqdm(train_loader, leave=False):
        xb = xb.to(device); yb = yb.to(device)
        xb, y_a, y_b, lam = mixup_data(xb, yb, alpha=0.2)
        optimizer.zero_grad()
        out = model(xb)
        if y_a is None:
            loss = criterion(out, yb)
        else:
            loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * xb.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
    train_acc = 100.0 * correct / total

    # validation
    model.eval()
    val_loss = 0.0
    correct_v = 0; total_v = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device); yb = yb.to(device)
            out = model(xb)
            val_loss += criterion(out, yb).item() * xb.size(0)
            preds = out.argmax(dim=1)
            correct_v += (preds == yb).sum().item()
            total_v += xb.size(0)
    val_acc = 100.0 * correct_v / total_v

    print(f"Ep {ep}/{epochs} train_acc {train_acc:.2f}% val_acc {val_acc:.2f}% loss {running_loss/total:.4f}")
    if val_acc > best:
        best = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best:", best)
    if best >= 85.0:
        print("Reached target 85% -> stopping.")
        break


100%|██████████| 170M/170M [00:03<00:00, 47.4MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Ep 1/300 train_acc 22.44% val_acc 42.92% loss 1.9943
Saved best: 42.92


Ep 2/300 train_acc 30.11% val_acc 50.51% loss 1.7846
Saved best: 50.51


Ep 3/300 train_acc 30.36% val_acc 56.48% loss 1.7069
Saved best: 56.48


Ep 4/300 train_acc 30.88% val_acc 58.54% loss 1.6457
Saved best: 58.54


...

Ep 236/300 train_acc 47.40% val_acc 82.67% loss 1.1327


Ep 237/300 train_acc 51.01% val_acc 84.64% loss 1.1400
Saved best: 84.64


Ep 238/300 train_acc 44.97% val_acc 82.12% loss 1.1159


Ep 239/300 train_acc 50.60% val_acc 83.03% loss 1.1368


Ep 240/300 train_acc 48.64% val_acc 86.25% loss 1.0959
Saved best: 86.25
Reached target 85% -> stopping.
